In [17]:
# %load_ext autoreload
# %autoreload 2

import os
import sqlite3
from glob import glob

import joblib
import pandas as pd
import numpy as np
import requests
from arch.univariate.base import ARCHModelResult
from config import settings
from data import SQLRepository
from model import GarchModel
from main import PredictIn, PredictOut

In [3]:
connection = sqlite3.connect(database=settings.db_name, check_same_thread=False)
repo = SQLRepository(connection=connection)

print("repo type:", type(repo))
print("repo.connection type:", type(repo.connection))

repo type: <class 'data.SQLRepository'>
repo.connection type: <class 'sqlite3.Connection'>


In [4]:

# Instantiate a `GarchModel`
gm_ambuja = GarchModel(ticker="AMBUJACEM.BSE", repo=repo, use_new_data=False)

# Does `gm_ambuja` have the correct attributes?
assert gm_ambuja.ticker == "AMBUJACEM.BSE"
assert gm_ambuja.repo == repo
assert not gm_ambuja.use_new_data
assert gm_ambuja.model_directory == settings.model_directory

In [5]:
# Instantiate `GarchModel`, use new data
model_shop = GarchModel(ticker="SHOPERSTOP.BSE", repo=repo, use_new_data=True)

# Check that model doesn't have `data` attribute yet
assert not hasattr(model_shop, "data")

# Wrangle data
model_shop.wrangle_data(n_observations=1000)

# Does model now have `data` attribute?
assert hasattr(model_shop, "data")

# Is the `data` a Series?
assert isinstance(model_shop.data, pd.Series)

# Is Series correct shape?
assert model_shop.data.shape == (1000,)

model_shop.data.head()

date
2022-03-24    4.645533
2022-03-25   -1.828597
2022-03-28    2.143178
2022-03-29    1.010656
2022-03-30    0.989668
Name: return, dtype: float64

In [6]:
# Instantiate `GarchModel`, use old data
model_shop = GarchModel(ticker="SHOPERSTOP.BSE", repo=repo, use_new_data=False)

# Wrangle data
model_shop.wrangle_data(n_observations=1000)

# Fit GARCH(1,1) model to data
model_shop.fit(p=1, q=1)

# Does `model_shop` have a `model` attribute now?
assert hasattr(model_shop, "model")

# Is model correct data type?
assert isinstance(model_shop.model, ARCHModelResult)

# Does model have correct parameters?
assert model_shop.model.params.index.tolist() == ["mu", "omega", "alpha[1]", "beta[1]"]

# Check model parameters
model_shop.model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2186.49
Distribution:                  Normal   AIC:                           4380.99
Method:            Maximum Likelihood   BIC:                           4400.62
                                        No. Observations:                 1000
Date:                Thu, Apr 09 2026   Df Residuals:                      999
Time:                        12:51:28   Df Model:                            1
                                Mean Model                               
=========================================================================
                  coef    std err          t      P>|t|  95.0% Conf. Int.
-------------------------------------------------------------------------
mu         -4.0409e-03  6.847e-02 -5.902e-02      0.953 [ -0.138,  0.130]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.9565      0.895      1.069      0.285    [ -0.797,  2.710]
alpha[1]       0.0610  3.593e-02      1.698  8.960e-02 [-9.429e-03,  0.131]
beta[1]        0.7377      0.197      3.744  1.810e-04    [  0.352,  1.124]
===========================================================================

Covariance estimator: robust
"""

In [7]:
# Generate prediction from `model_shop`
prediction = model_shop.predict_volatility(horizon=5)

# Is prediction a dictionary?
assert isinstance(prediction, dict)

# Are keys correct data type?
assert all(isinstance(k, str) for k in prediction.keys())

# Are values correct data type?
assert all(isinstance(v, float) for v in prediction.values())

prediction

{'2026-04-09T00:00:00': 2.1340988557989786,
 '2026-04-10T00:00:00': 2.1433459606550094,
 '2026-04-13T00:00:00': 2.1507027712596583,
 '2026-04-14T00:00:00': 2.156560395165515,
 '2026-04-15T00:00:00': 2.161227289357761}

In [8]:
# Save `model_shop` model, assign filename
filename = model_shop.dump()

# Is `filename` a string?
assert isinstance(filename, str)

# Does filename include ticker symbol?
assert model_shop.ticker in filename

# Does file exist?
assert os.path.exists(filename)

filename

'models\\2026-04-09T12-51-29.996423_SHOPERSTOP.BSE.pkl'

In [9]:
ticker ="SHOPERSTOP.BSE"
pattern= os.path.join(settings.model_directory, f"*{ticker}.pkl")
try:
    model_path=sorted(glob(pattern))[-1]
except IndexError:
    raise Exception(f"No model trained for '{ticker}'.")

In [10]:
def load(ticker):
    """Load latest model from model directory.

    Parameters
    ----------
    ticker : str
        Ticker symbol for which model was trained.

    Returns
    -------
    `ARCHModelResult`
    """
    # Create pattern for glob search
    pattern= os.path.join(settings.model_directory, f"*{ticker}.pkl")

    # Try to find path of latest model
    try:
        model_path = sorted(glob(pattern))[-1]

    # Handle possible `IndexError`
    except IndexError:
        raise Exception(f"No model trained for '{ticker}'.")

    # Load model
    model = joblib.load(model_path)

    # Return model
    return model

In [11]:
# Assign load output to `model`
model_shop = load(ticker="SHOPERSTOP.BSE")

# Does function return an `ARCHModelResult`
assert isinstance(model_shop, ARCHModelResult)

# Check model parameters
model_shop.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2186.49
Distribution:                  Normal   AIC:                           4380.99
Method:            Maximum Likelihood   BIC:                           4400.62
                                        No. Observations:                 1000
Date:                Thu, Apr 09 2026   Df Residuals:                      999
Time:                        12:51:28   Df Model:                            1
                                Mean Model                               
=========================================================================
                  coef    std err          t      P>|t|  95.0% Conf. Int.
-------------------------------------------------------------------------
mu         -4.0409e-03  6.847e-02 -5.902e-02      0.953 [ -0.138,  0.130]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.9565      0.895      1.069      0.285    [ -0.797,  2.710]
alpha[1]       0.0610  3.593e-02      1.698  8.960e-02 [-9.429e-03,  0.131]
beta[1]        0.7377      0.197      3.744  1.810e-04    [  0.352,  1.124]
===========================================================================

Covariance estimator: robust
"""

In [12]:
model_shop = GarchModel(ticker="SHOPERSTOP.BSE", repo=repo, use_new_data=False)

# Check that new `model_shop_test` doesn't have model attached
assert not hasattr(model_shop, "model")

# Load model
model_shop.load()

# Does `model_shop_test` have model attached?
assert hasattr(model_shop, "model")

model_shop.model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2186.49
Distribution:                  Normal   AIC:                           4380.99
Method:            Maximum Likelihood   BIC:                           4400.62
                                        No. Observations:                 1000
Date:                Thu, Apr 09 2026   Df Residuals:                      999
Time:                        12:51:28   Df Model:                            1
                                Mean Model                               
=========================================================================
                  coef    std err          t      P>|t|  95.0% Conf. Int.
-------------------------------------------------------------------------
mu         -4.0409e-03  6.847e-02 -5.902e-02      0.953 [ -0.138,  0.130]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.9565      0.895      1.069      0.285    [ -0.797,  2.710]
alpha[1]       0.0610  3.593e-02      1.698  8.960e-02 [-9.429e-03,  0.131]
beta[1]        0.7377      0.197      3.744  1.810e-04    [  0.352,  1.124]
===========================================================================

Covariance estimator: robust
"""

In [13]:
url = "http://localhost:8008/hello"
response = requests.get(url=url)

print("response code:", response.status_code)
response.json()

response code: 200


{'message': 'Hello world'}

In [14]:
from main import FitIn, FitOut

# Instantiate `FitIn`. Play with parameters.
fi = FitIn(
    ticker="SHOPERSTOP.BSE",
    use_new_data=True,
    n_observations=200,
    p=1,
    q=1
)
print(fi)

# Instantiate `FitOut`. Play with parameters.
fo = FitOut(
    ticker="SHOPERSTOP.BSE",
    use_new_data=True,
    n_observations=200,
    p=1,
    q=1,
    success=True,
    message="Our model is about to rock"
)
print(fo)

ticker='SHOPERSTOP.BSE' use_new_data=True n_observations=200 p=1 q=1
ticker='SHOPERSTOP.BSE' use_new_data=True n_observations=200 p=1 q=1 success=True message='Our model is about to rock'


In [15]:
from main import build_model

# Instantiate `GarchModel` with function
model_shop = build_model(ticker="SHOPERSTOP.BSE", use_new_data=False)

# Is `SQLRepository` attached to `model_shop`?
assert isinstance(model_shop.repo, SQLRepository)

# Is SQLite database attached to `SQLRepository`
assert isinstance(model_shop.repo.connection, sqlite3.Connection)

# Is `ticker` attribute correct?
assert model_shop.ticker == "SHOPERSTOP.BSE"

# Is `use_new_data` attribute correct?
assert not model_shop.use_new_data

model_shop

In [16]:
url = "http://localhost:8008/fit"

# Data to send to path
params = {
    "ticker": "SHOPERSTOP.BSE",
    "use_new_data": False,
    "n_observations": 2000,
    "p": 1,
    "q": 1
}
# Response of post request
response = requests.post(url=url, json=params)
# Inspect response
print("response code:", response.status_code)
response.json()

response code: 200


{'ticker': 'SHOPERSTOP.BSE',
 'use_new_data': False,
 'n_observations': 2000,
 'p': 1,
 'q': 1,
 'success': True,
 'message': "Trained and saved 'models\\2026-04-09T12-52-48.744937_SHOPERSTOP.BSE.pkl'. Metrics: AIC 9195.014103856964, BIC 9217.417713695131."}

In [18]:
pi = PredictIn(ticker="SHOPERSTOP.BSE", n_days=5)
print(pi)

po = PredictOut(
    ticker="SHOPERSTOP.BSE", n_days=5, success=True, forecast={}, message="success"
)
print(po)

ticker='SHOPERSTOP.BSE' n_days=5
ticker='SHOPERSTOP.BSE' n_days=5 success=True forecast={} message='success'


In [19]:
# URL of `/predict` path
url = "http://localhost:8008/predict"
# Data to send to path
params = {"ticker": "SHOPERSTOP.BSE", "n_days": 5}
# Response of post request
response = requests.post(url=url, json=params)
# Response JSON to be submitted to grader
submission = response.json()
# Inspect JSON
submission

{'ticker': 'SHOPERSTOP.BSE',
 'n_days': 5,
 'success': True,
 'forecast': {'2026-04-09T00:00:00': 2.4475257490509437,
  '2026-04-10T00:00:00': 2.4494697496669167,
  '2026-04-13T00:00:00': 2.451378464310764,
  '2026-04-14T00:00:00': 2.453252561334234,
  '2026-04-15T00:00:00': 2.4550926953887906},
 'message': ''}